# Analysis 1 — FC8 Weight Projection

For each somatic neuron `i`, `fc8.weight[:, i]` is a 1000-dim vector: positive entries = classes that neuron excites, negative = suppressed. Loaded directly from checkpoints (no forward passes needed).

Models compared:
- **2×2** (6 active branches): `alexnet_k1_ne4096_noproj_ddp_20260416_000308`
- **3×3×3** (39 active branches): `alexnet_k1_ne4096_3x3x3_noproj_ddp_20260506_230921`

In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

OUTPUTS = Path('/n/holylfs06/LABS/kempner_project_b/Lab/dendritic/adivm/dendritic-modeling/outputs')
FIG_DIR = Path('figures'); FIG_DIR.mkdir(exist_ok=True)

RUNS = {
    '2x2':   OUTPUTS / 'alexnet_k1_ne4096_noproj_ddp_20260416_000308',
    '3x3x3': OUTPUTS / 'alexnet_k1_ne4096_3x3x3_noproj_ddp_20260506_230921',
}
COLORS = {'2x2': '#2a78d6', '3x3x3': '#1baf7a'}
N_E = 4096
N_CLASSES = 1000

In [ ]:
def load_fc8_weights(run_dir):
    """Returns (N_E, 1000) weight matrix — the projection of each somatic neuron to classes."""
    ckpt = torch.load(run_dir / 'final_model.pt', map_location='cpu', weights_only=False)
    state = ckpt.get('model_state_dict', ckpt)
    W = state['decoder_network.layers.0.weight']  # (1000, N_E)
    b = state['decoder_network.layers.0.bias']    # (1000,)
    print(f"{run_dir.name}: W shape = {W.shape}")
    return W.T[:N_E].numpy().astype(np.float32), b.numpy()  # (N_E, 1000), (1000,)

weights = {k: load_fc8_weights(v) for k, v in RUNS.items()}
# weights[key] = (W_proj (N_E, 1000), bias (1000,))

alexnet_k1_ne4096_noproj_ddp_20260416_000308: W shape = torch.Size([1000, 4096])


## Figure 1 — Weight specialization: how peaked is each neuron's class vote vector?

For each neuron: `peak_ratio = max(|w|) / mean(|w|)`. High = one dominant class; low = broad tuning.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)

for ax, (label, (W, _)) in zip(axes, weights.items()):
    abs_W = np.abs(W)                                  # (N_E, 1000)
    peak_ratio = abs_W.max(axis=1) / (abs_W.mean(axis=1) + 1e-10)   # (N_E,)

    ax.hist(peak_ratio, bins=80, color=COLORS[label], edgecolor='none', alpha=0.85)
    ax.axvline(np.median(peak_ratio), color='#333', lw=1.2, ls='--', label=f'median={np.median(peak_ratio):.1f}')
    ax.set_xlabel('peak |w| / mean |w|  (class specialization)', fontsize=11)
    ax.set_ylabel('# neurons', fontsize=11)
    ax.set_title(f'{label}  (N_E={N_E})', fontsize=12)
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('FC8 weight specialization per neuron', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_weight_specialization.pdf', bbox_inches='tight')
plt.show()

## Figure 2 — Class coverage: how many neurons claim each class as top-1 weight target?

If every class is equally represented we'd expect ~4 neurons/class (4096/1000). Over-represented classes have many dedicated neurons.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, (label, (W, _)) in zip(axes, weights.items()):
    top_class = W.argmax(axis=1)           # (N_E,) — each neuron's top excitatory class
    counts    = np.bincount(top_class, minlength=N_CLASSES)   # (1000,)

    ax.hist(counts, bins=30, color=COLORS[label], edgecolor='none', alpha=0.85)
    ax.axvline(N_E / N_CLASSES, color='#333', lw=1.2, ls='--',
               label=f'expected={N_E/N_CLASSES:.1f}')
    ax.set_xlabel('# neurons with this class as top fc8 weight', fontsize=10)
    ax.set_ylabel('# classes', fontsize=11)
    ax.set_title(f'{label}', fontsize=12)
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    print(f"{label}: classes with 0 neurons = {(counts==0).sum()}  "
          f"max = {counts.max()} neurons on one class")

plt.suptitle('Class coverage by top fc8 weight (which class does each neuron most excite?)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_class_coverage.pdf', bbox_inches='tight')
plt.show()

## Figure 3 — Signed weight distribution: excitatory vs inhibitory class tuning

Histogram of all 4096×1000 weight values. Are most weights small/near-zero (sparse)? Is there asymmetry between positive (excites class) and negative (suppresses class) weights?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for label, (W, _) in weights.items():
    ax.hist(W.ravel(), bins=200, color=COLORS[label], alpha=0.6,
            label=label, density=True, histtype='stepfilled', edgecolor='none')

ax.axvline(0, color='#444', lw=1, ls=':')
ax.set_xlabel('fc8 weight value', fontsize=11)
ax.set_ylabel('density', fontsize=11)
ax.set_title('Distribution of all fc8 → somatic neuron weights', fontsize=12)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_weight_distribution.pdf', bbox_inches='tight')
plt.show()

## Figure 4 — Sorted weight heatmap (neuron × class)

Sort neurons by their argmax class, then plot a subsampled heatmap. Diagonal structure = neurons are class-specialized; noise = broad tuning.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, (W, _)) in zip(axes, weights.items()):
    # sort neurons by argmax class
    order     = np.argsort(W.argmax(axis=1))
    W_sorted  = W[order]                       # (N_E, 1000)

    # subsample rows for display (show every 4th neuron)
    W_disp = W_sorted[::4]                     # (1024, 1000)

    vmax = np.percentile(np.abs(W_disp), 99)
    im = ax.imshow(W_disp, aspect='auto', cmap='RdBu_r',
                   vmin=-vmax, vmax=vmax, interpolation='nearest')
    ax.set_xlabel('ImageNet class index', fontsize=10)
    ax.set_ylabel('Neuron (sorted by top class)', fontsize=10)
    ax.set_title(f'{label}', fontsize=12)
    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02, label='weight')

plt.suptitle('FC8 weight matrix — neurons sorted by top class\n(diagonal = class-specialized neurons)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_weight_heatmap.pdf', bbox_inches='tight')
plt.show()

## Figure 5 — Top-K class overlap: do weight top-class and activation top-class agree?

*(Requires `top_classes_by_weight.npy` and `top_classes_by_activation.npy` from the SLURM job — run after jobs complete.)*

In [ ]:
INTERP = OUTPUTS / 'interpretability_results'
TAGS   = {'2x2': 'fc7_2x2_ne4096', '3x3x3': 'fc7_3x3x3_ne4096'}

for label, tag in TAGS.items():
    wf = INTERP / tag / 'top_classes_by_weight.npy'
    af = INTERP / tag / 'top_classes_by_activation.npy'
    if not wf.exists() or not af.exists():
        print(f"{label}: SLURM job not yet complete — skipping")
        continue

    top_w = np.load(wf)   # (N_E, K)
    top_a = np.load(af)   # (N_E, K)
    K = top_w.shape[1]

    # Fraction of neurons where top-1 weight class == top-1 activation class
    top1_agree = (top_w[:, 0] == top_a[:, 0]).mean()
    # Fraction where top-1 weight class is in top-K activation classes
    topK_agree = np.array([top_w[i, 0] in top_a[i] for i in range(N_E)]).mean()

    print(f"{label}: top-1 agree = {top1_agree*100:.1f}%  "
          f"top-1 in top-{K} act = {topK_agree*100:.1f}%")